In [6]:
mamba install numpy
mamba install pandas
mamba install nltk
mamba install scikit-learn
mamba install xgboost
mamba install lightgbm

In [10]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/rzaheer/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from nltk.tokenize import sent_tokenize
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.metrics import f1_score, confusion_matrix, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from nltk.tokenize import word_tokenize
import re
import random
import warnings
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from numpy import mean, std
from sklearn.model_selection import GridSearchCV, cross_val_score,RandomizedSearchCV
import psutil, os
from joblib import dump, load
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import SGDClassifier
from sklearn.svm import LinearSVC
import xgboost as xgb
from sklearn.model_selection import StratifiedShuffleSplit

In [2]:
#### Importing the cleaned data into a Dataframe #####

df_american_movies = pd.read_csv("df_american_movies_post1969.csv")

df_american_movies.head()
#df_american_movies.shape # (8691, 7)

,Release Year,Title,Plot,PlotSummary,final_genre,final_director,final_cast
0,1970,Adam at Six A.M.,"The film revolves around Adam Gaines, a semant...","The film revolves around Adam Gaines, a semant...",Drama,Robert Scheerer,"Michael Douglas,Joe Don Baker,Charles Aidman,D..."
1,1970,The Adventurers,Set in the fictional Latin American country of...,Set in the fictional Latin American country of...,"Action,Adventure,Drama",Lewis Gilbert,"Charles Aznavour,Alan Badel,Thommy Berggren,Er..."
2,1970,Airport,Chicago is paralyzed by a snowstorm affecting ...,Chicago is paralyzed by a snowstorm affecting ...,"Action,Drama,Thriller",George Seaton,"Burt Lancaster,Dean Martin,George Kennedy,Van ..."
3,1970,Alex in Wonderland,Young director Alex Morrison feels compelled t...,Alex Morrison feels compelled to follow his re...,"Comedy,Drama",Paul Mazursky,"Donald Sutherland,Andre Philippe,Michael Lerne..."
4,1970,Angel Unchained,"Following a gang fight, biker Angel, calls it ...","Following a gang fight, biker Angel, calls it ...","Action,Drama,Thriller",Lee Madden,"Don Stroud,Luke Askew,Larry Bishop,T. Max Grah..."


In [3]:
# Split each line of the Plot into individual rows
#from nltk.tokenize import sent_tokenize
df_american_movies['Plot_chunks'] = df_american_movies['Plot'].apply(sent_tokenize)
df_american_movies = df_american_movies.explode('Plot_chunks', ignore_index=True)

#df_american_movies.shape # (222011, 8)
df_american_movies.head()

,Release Year,Title,Plot,PlotSummary,final_genre,final_director,final_cast,Plot_chunks
0,1970,Adam at Six A.M.,"The film revolves around Adam Gaines, a semant...","The film revolves around Adam Gaines, a semant...",Drama,Robert Scheerer,"Michael Douglas,Joe Don Baker,Charles Aidman,D...","The film revolves around Adam Gaines, a semant..."
1,1970,Adam at Six A.M.,"The film revolves around Adam Gaines, a semant...","The film revolves around Adam Gaines, a semant...",Drama,Robert Scheerer,"Michael Douglas,Joe Don Baker,Charles Aidman,D...",He becomes complacent in his life and hears ab...
2,1970,Adam at Six A.M.,"The film revolves around Adam Gaines, a semant...","The film revolves around Adam Gaines, a semant...",Drama,Robert Scheerer,"Michael Douglas,Joe Don Baker,Charles Aidman,D...",He drives cross country to attend the funeral ...
3,1970,Adam at Six A.M.,"The film revolves around Adam Gaines, a semant...","The film revolves around Adam Gaines, a semant...",Drama,Robert Scheerer,"Michael Douglas,Joe Don Baker,Charles Aidman,D...","He meets Jerri Jo Hopper, and falls in love, a..."
4,1970,Adam at Six A.M.,"The film revolves around Adam Gaines, a semant...","The film revolves around Adam Gaines, a semant...",Drama,Robert Scheerer,"Michael Douglas,Joe Don Baker,Charles Aidman,D...",He then must decide what direction he wants hi...


In [4]:
### Feature Engineering ####
# Creating year_before_2000_YN feature
df_american_movies['released_before_2000'] = (df_american_movies['Release Year'] < 2000).astype(int)

# combined_text: Creating a column that combines Plot_chunks, final_cast, final_genre, final_director
df_american_movies['combined_text'] = (
    df_american_movies['Plot_chunks']   + " " +
    df_american_movies['final_cast']    + " " +
    df_american_movies['final_genre']   + " " +
    df_american_movies['final_director']
)

# Approach #
## Combining text columns ###
### Training: X = plot_chunk + cast + genre + director ,  year_before_2000_YN
### Training: Y = Title 

## Testing data needs to be further synthesized to mimic the user inputs ###

### using X_test, create X_test1, X_test2, X_test3, X_test4, X_test5 #
    # X_test1 : Only takes plot_chunks , year_before_2000_YN =2
    # X_test2: plot_chunks , value for year_before_2000_YN
    # X_test3: plot_chunks + one genre , value for year_before_2000_YN
    # X_test4: plot_chunks + one genre + first actor , value for year_before_2000_YN
    # X_test5: plot_chunks + one genre + first actor + one director , value for year_before_2000_YN



In [5]:
### Creating data for predictor variables (X) and target variables (y)
X = df_american_movies[['Plot_chunks','final_genre','final_cast','final_director','combined_text','released_before_2000']]
y = df_american_movies.Title

In [6]:
### Removing movies with fewer than 5 entries in the dataset ###

min_samples = 5  # because 80-20 split can ensure at least 1 in test 
counts = df_american_movies.Title.value_counts()
valid_labels = counts[counts >= min_samples].index

mask = df_american_movies.Title.isin(valid_labels)
X_filtered = X.loc[mask]
y_filtered = y.loc[mask]



In [7]:
### Splitting the dataset into train, dev, and test###

RANDOM_SEED = 42
TRAIN_SIZE = .8
TEST_SIZE = .2

# Applying Train, dev, test split
X_train, X_test, y_train, y_test = train_test_split( X_filtered, y_filtered, test_size= TEST_SIZE, random_state=RANDOM_SEED,stratify=y_filtered)

# Dataframe for training
train_df = pd.concat([X_train, y_train], axis=1)

# Dataframe for testing 
test_df = pd.concat([X_test, y_test], axis=1)

## Keeping only what is needed for trainig , dev and test
# train - combined_text, released_before_2000, Title
train_df.drop(columns=["Plot_chunks","final_genre","final_cast","final_director"], inplace=True)

#train_df.shape # (175576, 3)

In [8]:
#### Creating 5 test dataset that mimics 5 user inputs to guess the movie 
## Only combined_text, released_before_2000, Title in each test dataset 

###### test1######
# test1 - Keep only Plot_chunks, 'released_before_2000'=2 ( fix value that doesnt exist in training), Title
test_df1 = test_df[['Plot_chunks','released_before_2000','Title']]
test_df1['released_before_2000']=2
# test1 - renaming Plot_chunks to combined_text
test_df1.rename(columns={'Plot_chunks': 'combined_text'}, inplace=True)

###### test2######
# test2: plot_chunks , value for year_before_2000_YN
test_df2 = test_df[['Plot_chunks','released_before_2000','Title']]
# test2 - renaming Plot_chunks to combined_text
test_df2.rename(columns={'Plot_chunks': 'combined_text'}, inplace=True)

###### test3######
# test 3 : plot_chunks + first genre , value for year_before_2000_YN
test_df3 = test_df[['Plot_chunks','final_genre','released_before_2000','Title']]

# test3: Randomly picking one of the genres to mimic user's input for genre question
test_df3['random_genre'] = test_df3['final_genre'].apply(
    lambda x: random.choice(re.split('[, ]+', x.strip()))
)

# test3: Combining Plot_chunks	and 'random_genre' to create combined text field at Q3 level 
test_df3['combined_text'] = test_df3['Plot_chunks']+" "+test_df3['random_genre']

# test3: Only keeping combined_text, released_before_2000, Title
test_df3 = test_df3[['combined_text','released_before_2000','Title']]

###### test4######
# test 4 : plot_chunks + one genre+ one actor, value for year_before_2000_YN

# adding final_cast to test_df3 to create test_df4 
test_df4 = test_df3.join(test_df['final_cast'], how='left')

# randomly choosing one of the casts from final_cast 

test_df4['random_cast'] = test_df4['final_cast'].apply(
    lambda x: random.choice([g.strip() for g in x.split(',')])
)

#adding random_cast to combined_text 
test_df4['combined_text']=test_df4['combined_text']+" "+test_df4['random_cast']

# dropping final_cast, random_cast
test_df4.drop(['final_cast', 'random_cast'], axis=1, inplace=True)

###### test5######
#plot_chunks + one genre+ one actor+one director, value for year_before_2000_YN, Title

# adding final_cast to test_df4 to create test_df5 
test_df5 = test_df4.join(test_df['final_director'], how='left')

# randomly choosing one of the directors from final_cast 

test_df5['random_director'] = test_df5['final_director'].apply(
    lambda x: random.choice([g.strip() for g in x.split(',')])
)

#adding random_cast to combined_text 
test_df5['combined_text']=test_df5['combined_text']+" "+test_df5['random_director']

# dropping final_cast, random_cast
test_df5.drop(['final_director', 'random_director'], axis=1, inplace=True)

#test_df1.shape # (43895, 3)
#test_df2.shape # (43895, 3)
#test_df3.shape # (43895, 3)
#test_df4.shape # (43895, 3)
#test_df5.shape # (43895, 3)

/tmp/ipykernel_2865219/3959181789.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df1['released_before_2000']=2
/tmp/ipykernel_2865219/3959181789.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df1.rename(columns={'Plot_chunks': 'combined_text'}, inplace=True)
/tmp/ipykernel_2865219/3959181789.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df2.rename(

In [8]:
# Saving test dfs into csv files
# Export to CSV
#test_df1.to_csv("test_df1.csv", index=False)
#test_df2.to_csv("test_df2.csv", index=False)
#test_df3.to_csv("test_df3.csv", index=False)
#test_df4.to_csv("test_df4.csv", index=False)
#test_df5.to_csv("test_df5.csv", index=False)


In [9]:
#### tf-idf vectorization: tokenization + stopword removal ####

#from cuml.feature_extraction.text import TfidfVectorizer

vectorizer_text = TfidfVectorizer(min_df =5,stop_words='english',max_features=3000)

X_train_combText = vectorizer_text.fit_transform(train_df.combined_text)
X_test1_combtext = vectorizer_text.transform(test_df1.combined_text)
X_test2_combtext = vectorizer_text.transform(test_df2.combined_text)
X_test3_combtext = vectorizer_text.transform(test_df3.combined_text)
X_test4_combtext = vectorizer_text.transform(test_df4.combined_text)
X_test5_combtext = vectorizer_text.transform(test_df5.combined_text)

print(type(X_train_combText),X_train_combText.shape, X_test1_combtext.shape,X_test2_combtext.shape, X_test3_combtext.shape,X_test4_combtext.shape,X_test5_combtext.shape)

<class 'scipy.sparse._csr.csr_matrix'> (175576, 3000) (43895, 3000) (43895, 3000) (43895, 3000) (43895, 3000) (43895, 3000)


In [10]:
# Coverting y values into a list 
y_train = list(train_df.Title)
y_test1 = list(test_df1.Title)
y_test2 = list(test_df2.Title)
y_test3 = list(test_df3.Title)
y_test4 = list(test_df4.Title)
y_test5 = list(test_df5.Title)

In [11]:
### y encoding - needed for xgboost and LinearSVC ### 

# -------------------------
# Ensure enough samples per class
# -------------------------
min_samples = 3  # because cv=3
counts = train_df.Title.value_counts()
valid_labels = counts[counts >= min_samples].index

mask = train_df.Title.isin(valid_labels)
X_train_filtered = train_df.loc[mask, ["combined_text", "released_before_2000"]]
y_train_filtered = train_df.Title.loc[mask]

# Encode labels AFTER filtering
le_wrapper = LabelEncoder()
y_train_encoded = le_wrapper.fit_transform(y_train_filtered)

# convert test Titles

class_to_int = {cls: i for i, cls in enumerate(le_wrapper.classes_)}

# Vectorized conversion with np.where
y_test1_encoded = np.array([
    class_to_int.get(label, -1)  # return -1 if not found
    for label in y_test1
])


y_test2_encoded = np.array([
    class_to_int.get(label, -1)  # return -1 if not found
    for label in y_test2
])

y_test3_encoded = np.array([
    class_to_int.get(label, -1)  # return -1 if not found
    for label in y_test3
])

y_test4_encoded = np.array([
    class_to_int.get(label, -1)  # return -1 if not found
    for label in y_test4
])

y_test5_encoded = np.array([
    class_to_int.get(label, -1)  # return -1 if not found
    for label in y_test5
])

# Saving the encoder
dump(le_wrapper, "LE_wrapper_hypertuned.joblib")

['LE_wrapper.joblib']

In [ ]:
### Creating a Dummy Model to set a baseline ###

## Dummy model 

RANDOM_SEED = 42

#equal probability to each label
uniform_clf = DummyClassifier(strategy='uniform', random_state = RANDOM_SEED)

preprocessor = ColumnTransformer(
    transformers=[
        ("tfidf", vectorizer_text, "combined_text"),
        ("year", "passthrough", ["released_before_2000"])
    ]
)

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("clf_movie", uniform_clf)
])

pipeline.fit(train_df[["combined_text", "released_before_2000"]], y_train)

# Saving the model as DummyModel.joblib
dump(pipeline, "DummyModel.joblib")


In [ ]:
### Building the best xgboost model with hyperparameter tuning ### 

models = {

    "Xgboost (Tree-Based)": {
       "model": xgb.XGBClassifier(
    objective="multi:softmax",  # or "binary:logistic" for binary classification
    eval_metric="mlogloss",     # or "logloss" for binary
    n_jobs= 1,                  # use 4 CPU cores
    tree_method="hist"          # fast & memory-efficient for sparse data
                            ),
        "params": {"max_depth": [6, 10]}
    }
}


# --------------------------
# CV + Hyperparameter Tuning
# --------------------------
cv_folds = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
scoring_metrics = ['accuracy']

for name, mp in models.items():
    print(f"\n--- {name} ---")
    
    # Preprocessor to vectorize text
    preprocessor_xgboost  = ColumnTransformer(
    transformers=[
        ("tfidf", vectorizer_text, "combined_text"),
        ("year", "passthrough", ["released_before_2000"]) # no transformation to "released_before_2000"
    ]
)

    #Pipeline that transforms the input and feed it into the model 
    pipeline = Pipeline([
    ("preprocess", preprocessor_xgboost),
    ("model", mp["model"])
    ])
    # Prefix model params with the pipeline step name
    param_grid = {f"model__{k}": v for k, v in mp["params"].items()}


    ###Grid Search
   
    #grid = GridSearchCV(pipeline, param_grid, cv=cv_folds, scoring='accuracy', n_jobs=-1)
    grid = RandomizedSearchCV(
    pipeline,
    param_grid,
    n_iter=2,             # number of random hypertuning combinations to try: 3 out of 4. 
    cv=cv_folds,
    scoring="accuracy",
    n_jobs= -1,             # parallelize across CPU cores
    random_state=42,
    verbose=2)
    
    # Memory usage in MB
    process = psutil.Process(os.getpid())
    print("Memory usage:", process.memory_info().rss / 1024**2, "MB")

    # CPU usage (percentage)
    print("CPU usage:", psutil.cpu_percent(interval=1), "%")

    grid.fit(X_train_filtered, y_train_encoded)

    
    # Memory usage in MB
    process = psutil.Process(os.getpid())
    print("Memory usage:", process.memory_info().rss / 1024**2, "MB")

    # CPU usage (percentage)
    print("CPU usage:", psutil.cpu_percent(interval=1), "%")

    print("Best Params:", grid.best_params_)
    
    best_model = grid.best_estimator_
 
    # Saving the model
    dump(best_model, "Xgboost_Hypertuned.joblib")
    #dump(le_wrapper, "LE_wrapper_xgboost.joblib")

    # Exporting Hyperparameter CSV
    results_Xgboost = pd.DataFrame(grid.cv_results_)
    # Export to CSV
    results_Xgboost.to_csv("hypertuning_results_Xgboost.csv", index=False)
    
    
   # Xgboost accuracy 
    
    score1 = best_model.score(test_df1[["combined_text", "released_before_2000"]], y_test1_encoded)
    score2 = best_model.score(test_df2[["combined_text", "released_before_2000"]], y_test2_encoded)
    score3 = best_model.score(test_df3[["combined_text", "released_before_2000"]], y_test3_encoded)
    score4 = best_model.score(test_df4[["combined_text", "released_before_2000"]], y_test4_encoded)
    score5 = best_model.score(test_df5[["combined_text", "released_before_2000"]], y_test5_encoded)

    print(f"Accuracy on Test1: {score1:.4f}")
    print(f"Accuracy on Test2: {score2:.4f}")
    print(f"Accuracy on Test3: {score3:.4f}")
    print(f"Accuracy on Test4: {score4:.4f}")
    print(f"Accuracy on Test5: {score5:.4f}")
    
    


--- Xgboost (Tree-Based) ---
Memory usage: 655.90234375 MB
CPU usage: 0.1 %
Fitting 3 folds for each of 2 candidates, totalling 6 fits
[CV] END ...............................model__max_depth=10; total time=394.0min
[CV] END ...............................model__max_depth=10; total time=437.5min
[CV] END ................................model__max_depth=6; total time=438.4min
[CV] END ................................model__max_depth=6; total time=458.0min
[CV] END ...............................model__max_depth=10; total time=483.9min
[CV] END ................................model__max_depth=6; total time=489.4min
Memory usage: 3548.16796875 MB
CPU usage: 0.1 %
Best Params: {'model__max_depth': 10}


In [16]:
### Building the best LinearSVC model with hyperparameter tuning ### 

models = {
    "LinearSVC": {
        "model": LinearSVC(max_iter=1000),   # higher max_iter helps convergence
        "params": {
            "C": [0.01, 0.1, 1]   # regularization strength
        }
    }
}

## CV fold and Hypertuning 

cv_folds = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
scoring_metrics = ['accuracy']

for name, mp in models.items():
    print(f"\n--- {name} ---")
    
    # Preprocessor to vectorize text
    preprocessor_linearSVC  = ColumnTransformer(
    transformers=[
        ("tfidf", vectorizer_text, "combined_text"),
        ("year", "passthrough", ["released_before_2000"]) # no transformation to "released_before_2000"
    ]
    )

    #Pipeline that transforms the input and feed it into the model 
    pipeline = Pipeline([
    ("preprocess", preprocessor_linearSVC),
    ("model", mp["model"])
    ])
    # Prefix model params with the pipeline step name
    param_grid = {f"model__{k}": v for k, v in mp["params"].items()}

    
    ###Grid Search####

    #grid = GridSearchCV(pipeline, param_grid, cv=cv_folds, scoring='accuracy', n_jobs=-1)
    grid = RandomizedSearchCV(
    pipeline,
    param_grid,
    n_iter=3,             # number of random combinations to try
    cv=cv_folds,
    scoring="accuracy",
    n_jobs=-1,             # parallelize across CPU cores
    random_state=42,
    verbose=2)

    # Memory usage in MB
    process = psutil.Process(os.getpid())
    print("Memory usage:", process.memory_info().rss / 1024**2, "MB")

    # CPU usage (percentage)
    print("CPU usage:", psutil.cpu_percent(interval=1), "%")
    
    # Fitting the model 
    grid.fit(X_train_filtered, y_train_encoded)
    
    # Memory usage in MB
    process = psutil.Process(os.getpid())
    print("Memory usage:", process.memory_info().rss / 1024**2, "MB")

    # CPU usage (percentage)
    print("CPU usage:", psutil.cpu_percent(interval=1), "%")
    # Fitting the model 
    
    best_model = grid.best_estimator_

    print("Best Params:", grid.best_params_)

    # Saving the model  
    dump(best_model, "LinearSVC_hypertuned.joblib")
    #dump(le_wrapper, "LE_wrapper_hypertuned.joblib")

    # Export Hyperparameter CSV
    results_LinearSVC = pd.DataFrame(grid.cv_results_)
    # Export to CSV
    results_LinearSVC.to_csv("hypertuning_results_LinearSVC.csv", index=False)
    

   # LinearSVC Accuracy
    
    score1 = best_model.score(test_df1[["combined_text", "released_before_2000"]], y_test1_encoded)
    score2 = best_model.score(test_df2[["combined_text", "released_before_2000"]], y_test2_encoded)
    score3 = best_model.score(test_df3[["combined_text", "released_before_2000"]], y_test3_encoded)
    score4 = best_model.score(test_df4[["combined_text", "released_before_2000"]], y_test4_encoded)
    score5 = best_model.score(test_df5[["combined_text", "released_before_2000"]], y_test5_encoded)

    print(f"Accuracy on Test1: {score1:.4f}")
    print(f"Accuracy on Test2: {score2:.4f}")
    print(f"Accuracy on Test3: {score3:.4f}")
    print(f"Accuracy on Test4: {score4:.4f}")
    print(f"Accuracy on Test5: {score5:.4f}")
    



--- LinearSVC ---
Memory usage: 650.93359375 MB
CPU usage: 1.7 %
Fitting 3 folds for each of 3 candidates, totalling 9 fits
[CV] END .......................................model__C=0.1; total time=59.4min
[CV] END .......................................model__C=0.1; total time=62.3min
[CV] END .......................................model__C=0.1; total time=62.5min
[CV] END .........................................model__C=1; total time=64.4min
[CV] END .........................................model__C=1; total time=65.4min
[CV] END .........................................model__C=1; total time=65.8min
[CV] END ......................................model__C=0.01; total time=77.2min
[CV] END ......................................model__C=0.01; total time=79.2min
[CV] END ......................................model__C=0.01; total time=80.1min
Memory usage: 856.54296875 MB
CPU usage: 0.1 %
Best Params: {'model__C': 1}
Accuracy on Test1: 0.0042
Accuracy on Test2: 0.0779
Accuracy on Test3:

In [ ]:
### Building the best SGDClassifier model with hyperparameter tuning ### 


#--- Hypertuned
models = {
    "SGDClassifier": {
        "model": SGDClassifier(loss="log_loss",          # logistic regression
    max_iter=1000,
    tol=1e-3, n_jobs=1),   # higher max_iter helps convergence
        "params": {
        "alpha": [0.0001, 0.001, 0.01]   # regularization strength
        }
    }
}

## CV fold and Hypertuning 

cv_folds = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
scoring_metrics = ['accuracy']

for name, mp in models.items():
    print(f"\n--- {name} ---")
    
    # Preprocessor to vectorize text
    preprocessor_SGDClassifier  = ColumnTransformer(
    transformers=[
        ("tfidf", vectorizer_text, "combined_text"),
        ("year", "passthrough", ["released_before_2000"]) # no transformation to "released_before_2000"
    ]
    )

    #Pipeline that transforms the input and feed it into the model 
    pipeline = Pipeline([
    ("preprocess", preprocessor_SGDClassifier),
    ("model", mp["model"])
    ])
    # Prefix model params with the pipeline step name
    param_grid = {f"model__{k}": v for k, v in mp["params"].items()}

    
    ###Grid Search####

    #grid = GridSearchCV(pipeline, param_grid, cv=cv_folds, scoring='accuracy', n_jobs=-1)
    grid = RandomizedSearchCV(
    pipeline,
    param_grid,
    n_iter=3,             # number of random combinations to try
    cv=cv_folds,
    scoring="accuracy",
    n_jobs= 30,             # parallelize across CPU cores
    random_state=42,
    verbose=2)
    

    # Fitting the model 
    grid.fit(X_train_filtered, y_train)

    # Saving hypertuning summary 
    results_SGDClassifier = pd.DataFrame(grid.cv_results_)
    
    # Exporting hypertuning summary  to CSV
    results_SGDClassifier.to_csv("hypertune_SGDClassifier.csv", index=False))
    
    # Storing best model 
    best_model = grid.best_estimator_

    # Saving the model  
    dump(best_model, "SGDClassifier_Hypertuned.joblib")
    #dump(le_wrapper, "LE_wrapper_hypertuned.joblib")
    
    print("Best Params:", grid.best_params_)

    #SGDClassifier Accuracy

    score1 = best_model.score(test_df1[["combined_text", "released_before_2000"]], y_test1)
    score2 = best_model.score(test_df2[["combined_text", "released_before_2000"]], y_test2)
    score3 = best_model.score(test_df3[["combined_text", "released_before_2000"]], y_test3)
    score4 = best_model.score(test_df4[["combined_text", "released_before_2000"]], y_test4)
    score5 = best_model.score(test_df5[["combined_text", "released_before_2000"]], y_test5)

    print(f"Accuracy on Test1: {score1:.4f}")
    print(f"Accuracy on Test2: {score2:.4f}")
    print(f"Accuracy on Test3: {score3:.4f}")
    print(f"Accuracy on Test4: {score4:.4f}")
    print(f"Accuracy on Test5: {score5:.4f}")



In [ ]:
#--- SGDClassifier ---
#Fitting 3 folds for each of 3 candidates, totalling 9 fits
#[CV] END ................................model__alpha=0.0001; total time=27.8min
#[CV] END ................................model__alpha=0.0001; total time=27.9min
#[CV] END ................................model__alpha=0.0001; total time=28.0min
#[CV] END .................................model__alpha=0.001; total time=31.7min
#[CV] END .................................model__alpha=0.001; total time=32.0min
#[CV] END .................................model__alpha=0.001; total time=32.8min
#[CV] END ..................................model__alpha=0.01; total time=50.6min
#[CV] END ..................................model__alpha=0.01; total time=51.6min
#[CV] END ..................................model__alpha=0.01; total time=52.2min
#Best Params: {'model__alpha': 0.0001}
#Accuracy on Test1: 0.0030
#Accuracy on Test2: 0.0160
#Accuracy on Test3: 0.0247
#Accuracy on Test4: 0.1462
#Accuracy on Test5: 0.2856

## Calculating Evaluation Metrics on 5-fold cross vadiation samples

### Mean Accuracy score, standard deviation of Accuracy 
### Mean F1-score, standard deviation of F1-score

In [ ]:
### Dummy Model ###

# Splitting the entire data into 5 different train/test splits
splits = StratifiedShuffleSplit(n_splits=5, test_size=0.2, random_state=42)

#Accuracy
scores1 = []
scores2 =[]
scores3=[]
scores4=[]
scores5=[]

#F1-score
scores1_f1 = []
scores2_f1 =[]
scores3_f1=[]
scores4_f1=[]
scores5_f1=[]

for train_idx, test_idx in splits.split(X_filtered , y_filtered):
    X_train, X_test = X_filtered.iloc[train_idx], X_filtered.iloc[test_idx]
    y_train, y_test = y_filtered.iloc[train_idx], y_filtered.iloc[test_idx]  # y_filtered.iloc[train_idx]

    # Calculating X_test1, X_test2, X_test3,X_test4, X_test5 & corresponding Ys
    train_df = pd.concat([X_train, y_train], axis=1)

    #dev_df = pd.concat([X_dev, y_dev], axis=1)
    
    test_df = pd.concat([X_test, y_test], axis=1)
    
    ## Keep what is needed for trainig , dev and test
    # train - combined_text, released_before_2000, Title
    train_df.drop(columns=["Plot_chunks","final_genre","final_cast","final_director"], inplace=True)
    
    
    ###### test1######
    # test1 - Keep only Plot_chunks, 'released_before_2000'=2 ( fix value that doesnt exist in training), Title
    test_df1 = test_df[['Plot_chunks','released_before_2000','Title']]
    test_df1['released_before_2000']=2
    # test1 - renaming Plot_chunks to combined_text
    test_df1.rename(columns={'Plot_chunks': 'combined_text'}, inplace=True)
    
    ###### test2######
    # test2: plot_chunks , value for year_before_2000_YN
    test_df2 = test_df[['Plot_chunks','released_before_2000','Title']]
    # test2 - renaming Plot_chunks to combined_text
    test_df2.rename(columns={'Plot_chunks': 'combined_text'}, inplace=True)
    
    ###### test3######
    # test 3 : plot_chunks + first genre , value for year_before_2000_YN
    test_df3 = test_df[['Plot_chunks','final_genre','released_before_2000','Title']]
    
    # test3: Randomly picking one of the genres to mimic user's input for genre question
    test_df3['random_genre'] = test_df3['final_genre'].apply(
        lambda x: random.choice(re.split('[, ]+', x.strip()))
    )
    
    # test3: Combining Plot_chunks	and 'random_genre' to create combined text field at Q3 level 
    test_df3['combined_text'] = test_df3['Plot_chunks']+" "+test_df3['random_genre']
    
    # test3: Only keeping combined_text, released_before_2000, Title
    test_df3 = test_df3[['combined_text','released_before_2000','Title']]
    
    ###### test4######
    # test 4 : plot_chunks + one genre+ one actor, value for year_before_2000_YN
    
    # adding final_cast to test_df3 to create test_df4 
    test_df4 = test_df3.join(test_df['final_cast'], how='left')
    
    # randomly choosing one of the casts from final_cast 
    
    test_df4['random_cast'] = test_df4['final_cast'].apply(
        lambda x: random.choice([g.strip() for g in x.split(',')])
    )
    
    #adding random_cast to combined_text 
    test_df4['combined_text']=test_df4['combined_text']+" "+test_df4['random_cast']
    
    # dropping final_cast, random_cast
    test_df4.drop(['final_cast', 'random_cast'], axis=1, inplace=True)
    
    ###### test5######
    #plot_chunks + one genre+ one actor+one director, value for year_before_2000_YN, Title
    
    # adding final_cast to test_df4 to create test_df5 
    test_df5 = test_df4.join(test_df['final_director'], how='left')
    
    # randomly choosing one of the directors from final_cast 
    
    test_df5['random_director'] = test_df5['final_director'].apply(
        lambda x: random.choice([g.strip() for g in x.split(',')])
    )
    
    #adding random_cast to combined_text 
    test_df5['combined_text']=test_df5['combined_text']+" "+test_df5['random_director']
    
    # dropping final_cast, random_cast
    test_df5.drop(['final_director', 'random_director'], axis=1, inplace=True)

    # Coverting y values into a list 
    y_train = list(train_df.Title)
    y_test1 = list(test_df1.Title)
    y_test2 = list(test_df2.Title)
    y_test3 = list(test_df3.Title)
    y_test4 = list(test_df4.Title)
    y_test5 = list(test_df5.Title)

    # Load Dummy model models 
    model = load("DummyModel.joblib")  # Needs to be created in the same environment


    #Make predictions
    # --- Make predictions ---
    y_pred1 = model.predict(test_df1[["combined_text", "released_before_2000"]])
    y_pred2 = model.predict(test_df2[["combined_text", "released_before_2000"]])
    y_pred3 = model.predict(test_df3[["combined_text", "released_before_2000"]])
    y_pred4 = model.predict(test_df4[["combined_text", "released_before_2000"]])
    y_pred5 = model.predict(test_df5[["combined_text", "released_before_2000"]])

    #Accuracy
    scores1.append(accuracy_score(y_test1, y_pred1))
    scores2.append(accuracy_score(y_test2, y_pred2))
    scores3.append(accuracy_score(y_test3, y_pred3))
    scores4.append(accuracy_score(y_test4, y_pred4))
    scores5.append(accuracy_score(y_test5, y_pred5))

    # LinearSVC - F1 score

    scores1_f1.append(f1_score(y_test1, y_pred1, average='weighted'))
    scores2_f1.append(f1_score(y_test2, y_pred2, average='weighted'))
    scores3_f1.append(f1_score(y_test3, y_pred3, average='weighted'))
    scores4_f1.append(f1_score(y_test4, y_pred4, average='weighted'))
    scores5_f1.append(f1_score(y_test5, y_pred5, average='weighted'))
    
    
    
print("Dummymodel Mean Scores across 5 different splits test1:", np.mean(scores1))
print("Dummymodel Std dev test1:", np.std(scores1))

print("Dummymodel Mean Scores across 5 different splits-test2:", np.mean(scores2))
print("Dummymodel Std dev test2:", np.std(scores2))

print("Dummymodel Mean Scores across 5 different splits-test3:", np.mean(scores3))
print("Dummymodel Std dev test3:", np.std(scores3))

print("Dummymodel Mean Scores across 5 different splits-test4:", np.mean(scores4))
print("Dummymodel Std dev test4:", np.std(scores4))

print("Dummymodel Mean Scores across 5 different splits-test5:", np.mean(scores5))
print("Dummymodel Std dev test5:", np.std(scores5))

## F1-score

print("Dummymodel Mean F1-Scores across 5 different splits test1:", np.mean(scores1_f1))
print("Dummymodel F1 Std dev test1:", np.std(scores1_f1))

print("Dummymodel Mean F1-Scores across 5 different splits-test2:", np.mean(scores2_f1))
print("Dummymodel F1 Std dev test2:", np.std(scores2_f1))

print("Dummymodel Mean F1-Scores across 5 different splits-test3:", np.mean(scores3_f1))
print("Dummymodel F1 Std dev test3:", np.std(scores3_f1))

print("Dummymodel Mean F1-Scores across 5 different splits-test4:", np.mean(scores4_f1))
print("Dummymodel F1 Std dev test4:", np.std(scores4_f1))

print("Dummymodel Mean F1-cores across 5 different splits-test5:", np.mean(scores5_f1))
print("Dummymodel F1 Std dev test5:", np.std(scores5_f1))


In [ ]:
#Dummymodel Mean Scores across 5 different splits test1: 0.00013215457528253737
#Dummymodel Std dev test1: 5.0745209285727504e-05
#Dummymodel Mean Scores across 5 different splits-test2: 0.00013215457528253737
#Dummymodel Std dev test2: 5.0745209285727504e-05
#Dummymodel Mean Scores across 5 different splits-test3: 0.00013215457528253737
#Dummymodel Std dev test3: 5.0745209285727504e-05
#Dummymodel Mean Scores across 5 different splits-test4: 0.00013215457528253737
#Dummymodel Std dev test4: 5.0745209285727504e-05
#Dummymodel Mean Scores across 5 different splits-test5: 0.00013215457528253737
#Dummymodel Std dev test5: 5.0745209285727504e-05
#Dummymodel Mean F1-Scores across 5 different splits test1: 0.00013380780108044645
#Dummymodel F1 Std dev test1: 5.796325541719984e-05
#Dummymodel Mean F1-Scores across 5 different splits-test2: 0.00013380780108044645
#Dummymodel F1 Std dev test2: 5.796325541719984e-05
#Dummymodel Mean F1-Scores across 5 different splits-test3: 0.00013380780108044645
#Dummymodel F1 Std dev test3: 5.796325541719984e-05
#Dummymodel Mean F1-Scores across 5 different splits-test4: 0.00013380780108044645
#Dummymodel F1 Std dev test4: 5.796325541719984e-05
#Dummymodel Mean F1-cores across 5 different splits-test5: 0.00013380780108044645
#Dummymodel F1 Std dev test5: 5.796325541719984e-05

In [ ]:
### LinearSVC ####

# Split the entire data into 5 different train/test splits
splits = StratifiedShuffleSplit(n_splits=5, test_size=0.2, random_state=42)

scores1 = []
scores2 =[]
scores3=[]
scores4=[]
scores5=[]

scores1_f1 = []
scores2_f1 =[]
scores3_f1=[]
scores4_f1=[]
scores5_f1=[]

for train_idx, test_idx in splits.split(X_filtered , y_filtered):
    X_train, X_test = X_filtered.iloc[train_idx], X_filtered.iloc[test_idx]
    y_train, y_test = y_filtered.iloc[train_idx], y_filtered.iloc[test_idx]  # y_filtered.iloc[train_idx]

    # Calculating X_test1, X_test2, X_test3,X_test4, X_test5 & corresponding Ys
    train_df = pd.concat([X_train, y_train], axis=1)

    #dev_df = pd.concat([X_dev, y_dev], axis=1)
    
    test_df = pd.concat([X_test, y_test], axis=1)
    
    ## Keep what is needed for trainig , dev and test
    # train - combined_text, released_before_2000, Title
    train_df.drop(columns=["Plot_chunks","final_genre","final_cast","final_director"], inplace=True)
    
    
    ###### test1######
    # test1 - Keep only Plot_chunks, 'released_before_2000'=2 ( fix value that doesnt exist in training), Title
    test_df1 = test_df[['Plot_chunks','released_before_2000','Title']]
    test_df1['released_before_2000']=2
    # test1 - renaming Plot_chunks to combined_text
    test_df1.rename(columns={'Plot_chunks': 'combined_text'}, inplace=True)
    
    ###### test2######
    # test2: plot_chunks , value for year_before_2000_YN
    test_df2 = test_df[['Plot_chunks','released_before_2000','Title']]
    # test2 - renaming Plot_chunks to combined_text
    test_df2.rename(columns={'Plot_chunks': 'combined_text'}, inplace=True)
    
    ###### test3######
    # test 3 : plot_chunks + first genre , value for year_before_2000_YN
    test_df3 = test_df[['Plot_chunks','final_genre','released_before_2000','Title']]
    
    # test3: Randomly picking one of the genres to mimic user's input for genre question
    test_df3['random_genre'] = test_df3['final_genre'].apply(
        lambda x: random.choice(re.split('[, ]+', x.strip()))
    )
    
    # test3: Combining Plot_chunks	and 'random_genre' to create combined text field at Q3 level 
    test_df3['combined_text'] = test_df3['Plot_chunks']+" "+test_df3['random_genre']
    
    # test3: Only keeping combined_text, released_before_2000, Title
    test_df3 = test_df3[['combined_text','released_before_2000','Title']]
    
    ###### test4######
    # test 4 : plot_chunks + one genre+ one actor, value for year_before_2000_YN
    
    # adding final_cast to test_df3 to create test_df4 
    test_df4 = test_df3.join(test_df['final_cast'], how='left')
    
    # randomly choosing one of the casts from final_cast 
    
    test_df4['random_cast'] = test_df4['final_cast'].apply(
        lambda x: random.choice([g.strip() for g in x.split(',')])
    )
    
    #adding random_cast to combined_text 
    test_df4['combined_text']=test_df4['combined_text']+" "+test_df4['random_cast']
    
    # dropping final_cast, random_cast
    test_df4.drop(['final_cast', 'random_cast'], axis=1, inplace=True)
    
    ###### test5######
    #plot_chunks + one genre+ one actor+one director, value for year_before_2000_YN, Title
    
    # adding final_cast to test_df4 to create test_df5 
    test_df5 = test_df4.join(test_df['final_director'], how='left')
    
    # randomly choosing one of the directors from final_cast 
    
    test_df5['random_director'] = test_df5['final_director'].apply(
        lambda x: random.choice([g.strip() for g in x.split(',')])
    )
    
    #adding random_cast to combined_text 
    test_df5['combined_text']=test_df5['combined_text']+" "+test_df5['random_director']
    
    # dropping final_cast, random_cast
    test_df5.drop(['final_director', 'random_director'], axis=1, inplace=True)

    # Coverting y values into a list 
    y_train = list(train_df.Title)
    y_test1 = list(test_df1.Title)
    y_test2 = list(test_df2.Title)
    y_test3 = list(test_df3.Title)
    y_test4 = list(test_df4.Title)
    y_test5 = list(test_df5.Title)

    # Load models and LE wrapper
    best_model_LinearSVC = load("LinearSVC_hypertuned.joblib")  # Needs to be created in the same environment
    le_wrapper = load("LE_wrapper_hypertuned.joblib")

    # encoding y train and test 
    y_train_encoded = le_wrapper.fit_transform(y_train)
    
    # converting test Titles
    
    class_to_int = {cls: i for i, cls in enumerate(le_wrapper.classes_)}
    
    # Vectorized conversion with np.where
    y_test1_encoded = np.array([
        class_to_int.get(label, -1)  # return -1 if not found
        for label in y_test1
    ])
    
    
    y_test2_encoded = np.array([
        class_to_int.get(label, -1)  # return -1 if not found
        for label in y_test2
    ])
    
    y_test3_encoded = np.array([
        class_to_int.get(label, -1)  # return -1 if not found
        for label in y_test3
    ])
    
    y_test4_encoded = np.array([
        class_to_int.get(label, -1)  # return -1 if not found
        for label in y_test4
    ])
    
    y_test5_encoded = np.array([
        class_to_int.get(label, -1)  # return -1 if not found
        for label in y_test5
    ])


    
    # LinearSVC - Accuracy 
    model = best_model_LinearSVC.fit(train_df[["combined_text", "released_before_2000"]], y_train_encoded)

    #Make predictions
    # --- Make predictions ---
    y_pred1 = model.predict(test_df1[["combined_text", "released_before_2000"]])
    y_pred2 = model.predict(test_df2[["combined_text", "released_before_2000"]])
    y_pred3 = model.predict(test_df3[["combined_text", "released_before_2000"]])
    y_pred4 = model.predict(test_df4[["combined_text", "released_before_2000"]])
    y_pred5 = model.predict(test_df5[["combined_text", "released_before_2000"]])

    #Accuracy
    scores1.append(accuracy_score(y_test1_encoded, y_pred1))
    scores2.append(accuracy_score(y_test2_encoded, y_pred2))
    scores3.append(accuracy_score(y_test3_encoded, y_pred3))
    scores4.append(accuracy_score(y_test4_encoded, y_pred4))
    scores5.append(accuracy_score(y_test5_encoded, y_pred5))

    # LinearSVC - F1 score

    scores1_f1.append(f1_score(y_test1_encoded, y_pred1, average='weighted'))
    scores2_f1.append(f1_score(y_test2_encoded, y_pred2, average='weighted'))
    scores3_f1.append(f1_score(y_test3_encoded, y_pred3, average='weighted'))
    scores4_f1.append(f1_score(y_test4_encoded, y_pred4, average='weighted'))
    scores5_f1.append(f1_score(y_test5_encoded, y_pred5, average='weighted'))
    
    
print("LinearSVC Mean Scores across 5 different splits test1:", np.mean(scores1))
print("LinearSVC Std dev test1:", np.std(scores1))

print("LinearSVC Mean Scores across 5 different splits-test2:", np.mean(scores2))
print("LinearSVC Std dev test2:", np.std(scores2))

print("LinearSVC Mean Scores across 5 different splits-test3:", np.mean(scores3))
print("LinearSVC Std dev test3:", np.std(scores3))

print("LinearSVC Mean Scores across 5 different splits-test4:", np.mean(scores4))
print("LinearSVC Std dev test4:", np.std(scores4))

print("LinearSVC Mean Scores across 5 different splits-test5:", np.mean(scores5))
print("LinearSVC Std dev test5:", np.std(scores5))

## F1-score

print("LinearSVC Mean F1-Scores across 5 different splits test1:", np.mean(scores1_f1))
print("LinearSVC F1 Std dev test1:", np.std(scores1_f1))

print("LinearSVC Mean F1-Scores across 5 different splits-test2:", np.mean(scores2_f1))
print("LinearSVC F1 Std dev test2:", np.std(scores2_f1))

print("LinearSVC Mean F1-Scores across 5 different splits-test3:", np.mean(scores3_f1))
print("LinearSVC F1 Std dev test3:", np.std(scores3_f1))

print("LinearSVC Mean F1-Scores across 5 different splits-test4:", np.mean(scores4_f1))
print("LinearSVC F1 Std dev test4:", np.std(scores4_f1))

print("LinearSVC Mean F1-cores across 5 different splits-test5:", np.mean(scores5_f1))
print("LinearSVC F1 Std dev test5:", np.std(scores5_f1))


In [ ]:
#LinearSVC Mean Scores across 5 different splits test1: 0.0038005833029529706
#LinearSVC Std dev test1: 0.0010634059817054469
#LinearSVC Mean Scores across 5 different splits-test2: 0.07838133430550491
#LinearSVC Std dev test2: 0.0011202385212652132
#LinearSVC Mean Scores across 5 different splits-test3: 0.11846062705067446
#LinearSVC Std dev test3: 0.0011620041737602984
#LinearSVC Mean Scores across 5 different splits-test4: 0.5154894276339774
#LinearSVC Std dev test4: 0.0011459338091339441
#LinearSVC Mean Scores across 5 different splits-test5: 0.8210216915785636
#LinearSVC Std dev test5: 0.0003358026508738702
#LinearSVC Mean F1-Scores across 5 different splits test1: 0.0036078564357410444
#LinearSVC F1 Std dev test1: 0.00108967653825264
#LinearSVC Mean F1-Scores across 5 different splits-test2: 0.07735631200575804
#LinearSVC F1 Std dev test2: 0.0007487236419283925
#LinearSVC Mean F1-Scores across 5 different splits-test3: 0.11584518500427539
#LinearSVC F1 Std dev test3: 0.0011671695439481934
#LinearSVC Mean F1-Scores across 5 different splits-test4: 0.5215700421926938
#LinearSVC F1 Std dev test4: 0.0016142549891188115
#LinearSVC Mean F1-cores across 5 different splits-test5: 0.8200154433735831
#LinearSVC F1 Std dev test5: 0.0003571317567403772

In [ ]:
### Xgboost ###

# Split the entire data into 5 different train/test splits
splits = StratifiedShuffleSplit(n_splits=5, test_size=0.2, random_state=42)

# Helper function to run one split
def run_split(train_idx, test_idx, X_filtered, y_filtered):
    X_train, X_test = X_filtered.iloc[train_idx], X_filtered.iloc[test_idx]
    y_train, y_test = y_filtered.iloc[train_idx], y_filtered.iloc[test_idx]

    train_df = pd.concat([X_train, y_train], axis=1)
    test_df = pd.concat([X_test, y_test], axis=1)

    ###### test1 ######
    test_df1 = test_df[['Plot_chunks', 'released_before_2000', 'Title']].copy()
    test_df1['released_before_2000'] = 2
    test_df1.rename(columns={'Plot_chunks': 'combined_text'}, inplace=True)

    ###### test2 ######
    test_df2 = test_df[['Plot_chunks', 'released_before_2000', 'Title']].copy()
    test_df2.rename(columns={'Plot_chunks': 'combined_text'}, inplace=True)

    ###### test3 ######
    test_df3 = test_df[['Plot_chunks', 'final_genre', 'released_before_2000', 'Title']].copy()
    test_df3['random_genre'] = test_df3['final_genre'].apply(lambda x: random.choice(re.split('[, ]+', x.strip())))
    test_df3['combined_text'] = test_df3['Plot_chunks'] + " " + test_df3['random_genre']
    test_df3 = test_df3[['combined_text', 'released_before_2000', 'Title']]

    ###### test4 ######
    test_df4 = test_df3.join(test_df['final_cast'], how='left')
    test_df4['random_cast'] = test_df4['final_cast'].apply(lambda x: random.choice([g.strip() for g in x.split(',')]))
    test_df4['combined_text'] = test_df4['combined_text'] + " " + test_df4['random_cast']
    test_df4.drop(['final_cast', 'random_cast'], axis=1, inplace=True)

    ###### test5 ######
    test_df5 = test_df4.join(test_df['final_director'], how='left')
    test_df5['random_director'] = test_df5['final_director'].apply(lambda x: random.choice([g.strip() for g in x.split(',')]))
    test_df5['combined_text'] = test_df5['combined_text'] + " " + test_df5['random_director']
    test_df5.drop(['final_director', 'random_director'], axis=1, inplace=True)

    # Clean up training data
    train_df.drop(columns=["Plot_chunks", "final_genre", "final_cast", "final_director"], inplace=True)

    # Prepare y
    y_train = list(train_df.Title)
    y_tests = [list(df.Title) for df in [test_df1, test_df2, test_df3, test_df4, test_df5]]

    # Load model and label encoder
    best_model_Xgboost = load("Xgboost_Hypertuned.joblib")
    le_wrapper = load("LE_wrapper_hypertuned.joblib")

    # Encode labels
    class_to_int = {cls: i for i, cls in enumerate(le_wrapper.classes_)}
    encode = lambda y: np.array([class_to_int.get(lbl, -1) for lbl in y])
    y_train_encoded = np.array([class_to_int.get(lbl, -1) for lbl in y_train])
    y_tests_encoded = [encode(y) for y in y_tests]

    # Fit model
    model = best_model_Xgboost.fit(train_df[["combined_text", "released_before_2000"]], y_train_encoded)

    # Evaluate each test dataset
    results = []
    for i, (df, y_true) in enumerate(zip([test_df1, test_df2, test_df3, test_df4, test_df5], y_tests_encoded), start=1):
        y_pred = model.predict(df[["combined_text", "released_before_2000"]])
        acc = accuracy_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred, average="weighted")
        results.append((acc, f1))

    return results


# Run each split in parallel
results = Parallel(n_jobs=-1, verbose=10)(
    delayed(run_split)(train_idx, test_idx, X_filtered, y_filtered)
    for train_idx, test_idx in splits.split(X_filtered, y_filtered)
)

# Transpose the results
# results is shape (5 splits × 5 testsets × 2 metrics)
acc_scores = [[r[i][0] for r in results] for i in range(5)]
f1_scores = [[r[i][1] for r in results] for i in range(5)]

# Print summary
for i in range(5):
    print(f"\nTest{i+1}:")
    print(f"  Accuracy mean: {np.mean(acc_scores[i]):.4f}, std: {np.std(acc_scores[i]):.4f}")
    print(f"  F1-score mean: {np.mean(f1_scores[i]):.4f}, std: {np.std(f1_scores[i]):.4f}")


In [ ]:
#Test1:
#  Accuracy mean: 0.0016, std: 0.0003
#  F1-score mean: 0.0016, std: 0.0002

#Test2:
#  Accuracy mean: 0.0015, std: 0.0003
#  F1-score mean: 0.0014, std: 0.0003

#Test3:
#  Accuracy mean: 0.0053, std: 0.0001
#  F1-score mean: 0.0036, std: 0.0002

#Test4:
#  Accuracy mean: 0.0472, std: 0.0014
#  F1-score mean: 0.0388, std: 0.0012

#Test5:
#  Accuracy mean: 0.1244, std: 0.0014
#  F1-score mean: 0.1077, std: 0.0008

In [ ]:
### SGDClassifier ###

# Split the entire data into 5 different train/test splits
splits = StratifiedShuffleSplit(n_splits=5, test_size=0.2, random_state=42)

#Accuracy
scores1 = []
scores2 =[]
scores3=[]
scores4=[]
scores5=[]

#F1-score
scores1_f1 = []
scores2_f1 =[]
scores3_f1=[]
scores4_f1=[]
scores5_f1=[]

for train_idx, test_idx in splits.split(X_filtered , y_filtered):
    X_train, X_test = X_filtered.iloc[train_idx], X_filtered.iloc[test_idx]
    y_train, y_test = y_filtered.iloc[train_idx], y_filtered.iloc[test_idx]  # y_filtered.iloc[train_idx]

    # Calculating X_test1, X_test2, X_test3,X_test4, X_test5 & corresponding Ys
    train_df = pd.concat([X_train, y_train], axis=1)

    #dev_df = pd.concat([X_dev, y_dev], axis=1)
    
    test_df = pd.concat([X_test, y_test], axis=1)
    
    ## Keep what is needed for trainig , dev and test
    # train - combined_text, released_before_2000, Title
    train_df.drop(columns=["Plot_chunks","final_genre","final_cast","final_director"], inplace=True)
    
    
    ###### test1######
    # test1 - Keep only Plot_chunks, 'released_before_2000'=2 ( fix value that doesnt exist in training), Title
    test_df1 = test_df[['Plot_chunks','released_before_2000','Title']]
    test_df1['released_before_2000']=2
    # test1 - renaming Plot_chunks to combined_text
    test_df1.rename(columns={'Plot_chunks': 'combined_text'}, inplace=True)
    
    ###### test2######
    # test2: plot_chunks , value for year_before_2000_YN
    test_df2 = test_df[['Plot_chunks','released_before_2000','Title']]
    # test2 - renaming Plot_chunks to combined_text
    test_df2.rename(columns={'Plot_chunks': 'combined_text'}, inplace=True)
    
    ###### test3######
    # test 3 : plot_chunks + first genre , value for year_before_2000_YN
    test_df3 = test_df[['Plot_chunks','final_genre','released_before_2000','Title']]
    
    # test3: Randomly picking one of the genres to mimic user's input for genre question
    test_df3['random_genre'] = test_df3['final_genre'].apply(
        lambda x: random.choice(re.split('[, ]+', x.strip()))
    )
    
    # test3: Combining Plot_chunks	and 'random_genre' to create combined text field at Q3 level 
    test_df3['combined_text'] = test_df3['Plot_chunks']+" "+test_df3['random_genre']
    
    # test3: Only keeping combined_text, released_before_2000, Title
    test_df3 = test_df3[['combined_text','released_before_2000','Title']]
    
    ###### test4######
    # test 4 : plot_chunks + one genre+ one actor, value for year_before_2000_YN
    
    # adding final_cast to test_df3 to create test_df4 
    test_df4 = test_df3.join(test_df['final_cast'], how='left')
    
    # randomly choosing one of the casts from final_cast 
    
    test_df4['random_cast'] = test_df4['final_cast'].apply(
        lambda x: random.choice([g.strip() for g in x.split(',')])
    )
    
    #adding random_cast to combined_text 
    test_df4['combined_text']=test_df4['combined_text']+" "+test_df4['random_cast']
    
    # dropping final_cast, random_cast
    test_df4.drop(['final_cast', 'random_cast'], axis=1, inplace=True)
    
    ###### test5######
    #plot_chunks + one genre+ one actor+one director, value for year_before_2000_YN, Title
    
    # adding final_cast to test_df4 to create test_df5 
    test_df5 = test_df4.join(test_df['final_director'], how='left')
    
    # randomly choosing one of the directors from final_cast 
    
    test_df5['random_director'] = test_df5['final_director'].apply(
        lambda x: random.choice([g.strip() for g in x.split(',')])
    )
    
    #adding random_cast to combined_text 
    test_df5['combined_text']=test_df5['combined_text']+" "+test_df5['random_director']
    
    # dropping final_cast, random_cast
    test_df5.drop(['final_director', 'random_director'], axis=1, inplace=True)

    # Coverting y values into a list 
    y_train = list(train_df.Title)
    y_test1 = list(test_df1.Title)
    y_test2 = list(test_df2.Title)
    y_test3 = list(test_df3.Title)
    y_test4 = list(test_df4.Title)
    y_test5 = list(test_df5.Title)

    # Load model
    model = load("SGDClassifier_Hypertuned.joblib")  # Needs to be created in the same environment
  

    #Make predictions
    # --- Make predictions ---
    y_pred1 = model.predict(test_df1[["combined_text", "released_before_2000"]])
    y_pred2 = model.predict(test_df2[["combined_text", "released_before_2000"]])
    y_pred3 = model.predict(test_df3[["combined_text", "released_before_2000"]])
    y_pred4 = model.predict(test_df4[["combined_text", "released_before_2000"]])
    y_pred5 = model.predict(test_df5[["combined_text", "released_before_2000"]])

    #Accuracy
    scores1.append(accuracy_score(y_test1, y_pred1))
    scores2.append(accuracy_score(y_test2, y_pred2))
    scores3.append(accuracy_score(y_test3, y_pred3))
    scores4.append(accuracy_score(y_test4, y_pred4))
    scores5.append(accuracy_score(y_test5, y_pred5))

    # LinearSVC - F1 score

    scores1_f1.append(f1_score(y_test1, y_pred1, average='weighted'))
    scores2_f1.append(f1_score(y_test2, y_pred2, average='weighted'))
    scores3_f1.append(f1_score(y_test3, y_pred3, average='weighted'))
    scores4_f1.append(f1_score(y_test4, y_pred4, average='weighted'))
    scores5_f1.append(f1_score(y_test5, y_pred5, average='weighted'))
    
    
    
print("SGDClassifier Mean Scores across 5 different splits test1:", np.mean(scores1))
print("SGDClassifier Std dev test1:", np.std(scores1))

print("SGDClassifier Mean Scores across 5 different splits-test2:", np.mean(scores2))
print("SGDClassifier Std dev test2:", np.std(scores2))

print("SGDClassifier Mean Scores across 5 different splits-test3:", np.mean(scores3))
print("SGDClassifier Std dev test3:", np.std(scores3))

print("SGDClassifier Mean Scores across 5 different splits-test4:", np.mean(scores4))
print("SGDClassifier Std dev test4:", np.std(scores4))

print("SGDClassifier Mean Scores across 5 different splits-test5:", np.mean(scores5))
print("SGDClassifier Std dev test5:", np.std(scores5))

## F1-score

print("SGDClassifier Mean F1-Scores across 5 different splits test1:", np.mean(scores1_f1))
print("SGDClassifier F1 Std dev test1:", np.std(scores1_f1))

print("SGDClassifier Mean F1-Scores across 5 different splits-test2:", np.mean(scores2_f1))
print("SGDClassifier F1 Std dev test2:", np.std(scores2_f1))

print("SGDClassifier Mean F1-Scores across 5 different splits-test3:", np.mean(scores3_f1))
print("SGDClassifier F1 Std dev test3:", np.std(scores3_f1))

print("SGDClassifier Mean F1-Scores across 5 different splits-test4:", np.mean(scores4_f1))
print("SGDClassifier F1 Std dev test4:", np.std(scores4_f1))

print("SGDClassifier Mean F1-cores across 5 different splits-test5:", np.mean(scores5_f1))
print("SGDClassifier F1 Std dev test5:", np.std(scores5_f1))



In [ ]:
#SGDClassifier Mean Scores across 5 different splits test1: 0.003513488880787459
#SGDClassifier Std dev test1: 0.00030344644211787277
#SGDClassifier Mean Scores across 5 different splits-test2: 0.020707254830477577
#SGDClassifier Std dev test2: 0.0019868652315712066
#SGDClassifier Mean Scores across 5 different splits-test3: 0.03137076193948232
#SGDClassifier Std dev test3: 0.0028813198965630354
#SGDClassifier Mean Scores across 5 different splits-test4: 0.1651932191031717
#SGDClassifier Std dev test4: 0.007629222129211606
#SGDClassifier Mean Scores across 5 different splits-test5: 0.3060472110827561
#SGDClassifier Std dev test5: 0.009234636427524716
#SGDClassifier Mean F1-Scores across 5 different splits test1: 0.0007307427917955791
#SGDClassifier F1 Std dev test1: 0.00014625517673913375
#SGDClassifier Mean F1-Scores across 5 different splits-test2: 0.015123652510100139
#SGDClassifier F1 Std dev test2: 0.002175896195110657
#SGDClassifier Mean F1-Scores across 5 different splits-test3: 0.022212080478731615
#SGDClassifier F1 Std dev test3: 0.003218892624219414
#SGDClassifier Mean F1-Scores across 5 different splits-test4: 0.14445615207499266
#SGDClassifier F1 Std dev test4: 0.009180989372552939
#SGDClassifier Mean F1-cores across 5 different splits-test5: 0.28677304137403403
#SGDClassifier F1 Std dev test5: 0.010724896050322848

## LinearSVC model analysis

### Model Summary
### Feature Importance
### Ablation Analysis
### Failure analysis

In [15]:
### Model Summary ###

def summarize_model(pipeline):
    print("Model Summary\n")
    print("Pipeline Steps:")
    for name, step in pipeline.named_steps.items():
        print(f" - {name}: {type(step).__name__}")
    
    print("\nModel Parameters:")
    clf = pipeline.named_steps['model']
    for param, val in clf.get_params().items():
        print(f"   {param}: {val}")
    
    if hasattr(clf, "coef_"):
        print("\nLearned Coefficients Shape:", clf.coef_.shape)
    
    print("\nPreprocessing Transformers:")
    print(pipeline.named_steps['preprocess'])

best_model = load("LinearSVC_hypertuned.joblib")
summarize_model(best_model)

Model Summary

Pipeline Steps:
 - preprocess: ColumnTransformer
 - model: LinearSVC

Model Parameters:
   C: 1
   class_weight: None
   dual: auto
   fit_intercept: True
   intercept_scaling: 1
   loss: squared_hinge
   max_iter: 1000
   multi_class: ovr
   penalty: l2
   random_state: None
   tol: 0.0001
   verbose: 0

Learned Coefficients Shape: (7590, 3001)

Preprocessing Transformers:
ColumnTransformer(transformers=[('tfidf',
                                 TfidfVectorizer(max_features=3000, min_df=5,
                                                 stop_words='english'),
                                 'combined_text'),
                                ('year', 'passthrough',
                                 ['released_before_2000'])])


In [14]:
### Feature Importance ###

# Loading trained pipeline
pipeline = load("LinearSVC_hypertuned.joblib")

# Step 1. Access components
preprocessor = pipeline.named_steps["preprocess"]
model = pipeline.named_steps["model"]

# Step 2. Extract feature names from ColumnTransformer
# 2a. TF-IDF feature names (from text)
tfidf = preprocessor.named_transformers_["tfidf"]
tfidf_features = tfidf.get_feature_names_out()

# 2b. Year feature name (passed through)
numeric_features = ["released_before_2000"]

# 2c. Combine in the same order as ColumnTransformer
feature_names = np.concatenate([tfidf_features, numeric_features])

# Step 3. Extract model coefficients
coefs = model.coef_

# Step 4.  multiclass - mean absolute value across classes:

importance = np.mean(np.abs(coefs), axis=0)

# Step 5. Combine and sort
importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importance
}).sort_values("importance", ascending=False)

# Step 6. Show top 10 features
print(importance_df.head(10))


                   feature  importance
1443                  john    0.177263
3000  released_before_2000    0.175387
1853               michael    0.173572
554                 comedy    0.166778
802                  drama    0.161219
687                  david    0.158665
2341               romance    0.158134
629                  crime    0.151596
20                  action    0.144805
2719              thriller    0.142053


In [13]:
### Ablation Analysis ###

# Load your best trained pipeline to reuse vectorizer and hyperparameters
best_pipeline = load("LinearSVC_hypertuned.joblib")
best_model = best_pipeline.named_steps["model"]

# Reuse vectorizer settings from the best model
vectorizer_text = best_pipeline.named_steps["preprocess"].named_transformers_["tfidf"]

le_wrapper = load("LE_wrapper_hypertuned.joblib")
#le_wrapper = load("LE_wrapper.joblib")

# convert test Titles

# encoding y train 
y_train_encoded = le_wrapper.fit_transform(y_train_filtered)
    
class_to_int = {cls: i for i, cls in enumerate(le_wrapper.classes_)}
    
# encoding  y_test5 
y_test5_encoded = np.array([
        class_to_int.get(label, -1)  # return -1 if not found
        for label in y_test5
    ])



def evaluate_variant(include_text=True, include_year=True):
    
    transformers = []
    if include_text:
        transformers.append(("tfidf", vectorizer_text, "combined_text"))
    if include_year:
        transformers.append(("year", "passthrough", ["released_before_2000"]))
    
    preprocessor = ColumnTransformer(transformers=transformers)
    
    # Use same model hyperparameters as the best one
    model_variant = LinearSVC(
        C=best_model.C,
        penalty=best_model.penalty,
        loss=best_model.loss,
        dual=best_model.dual,
        tol=best_model.tol,
        max_iter=best_model.max_iter
    )

    pipeline_variant = Pipeline([
        ("preprocess", preprocessor),
        ("model", model_variant)
    ])
 
    pipeline_variant.fit(train_df[["combined_text", "released_before_2000"]], y_train_encoded)
    
    #Prediction
    preds5 = pipeline_variant.predict(test_df5[["combined_text", "released_before_2000"]])

    #Accuracy
    score5 = accuracy_score(y_test5_encoded , preds5)
    #F1-score
    score5_f1 = f1_score(y_test5_encoded , preds5 ,average='weighted')
    

    return score5, score5_f1


# Evaluate variants
results = {
    "Full model": evaluate_variant(True, True),
    "Without text": evaluate_variant(False, True),
    "Without year": evaluate_variant(True, False)
}

# Show results
for name, acc in results.items():
    print(f"{name:20s}  Accuracy={acc[0]:.4f},  F1={acc[1]:.4f}")

Full model            Accuracy=0.8190,  F1=0.8172
Without text          Accuracy=0.0025,  F1=0.0000
Without year          Accuracy=0.8002,  F1=0.8002


In [38]:
#### failure Anlysis ####

## Failure Analysis - on Test_df5 

# Load saved LinearSVC pipeline and label encoder
pipeline = load("LinearSVC_hypertuned.joblib")      # this includes preprocessing + model
#le_wrapper = load("LE_wrapper_hypertuned.joblib")    # label encoder
le_wrapper = load("LE_wrapper_hypertuned.joblib")    # label encoder
#test_df5 = pd.read_csv("test_df5.csv") 

# Getting decision function (confidence scores)
# Shape: [n_samples, n_classes]
scores = pipeline.decision_function(test_df5[["combined_text", "released_before_2000"]])

# Finding the top predicted class index per row
pred_idx = np.argmax(scores, axis=1)

# Getting predicted label names
pred_labels = le_wrapper.inverse_transform(pred_idx)

# Getting confidence score (convert decision values to probabilities using softmax-like normalization)
confidences = np.max(scores, axis=1)
# normalize to 0–1 range for interpretability
confidences = (confidences - np.min(confidences)) / (np.max(confidences) - np.min(confidences))

# Compare predictions vs true labels
is_wrong = pred_labels != test_df5["Title"]

# Append to DataFrame
test_df5 = test_df5.copy()
test_df5["Predicted_Title"] = pred_labels
test_df5["Confidence_Score"] = confidences
test_df5["Prediction_Wrong"] = is_wrong

# Sort by lowest confidence (useful for failure analysis)
test_df5_sorted = test_df5.sort_values(by="Confidence_Score", ascending=True)

# dataframe to capture failed predictions
failures= test_df5_sorted[test_df5_sorted.Prediction_Wrong==True]

failures.head()


,combined_text,released_before_2000,Title,Predicted_Title,Confidence_Score,Prediction_Wrong
46057,"Before she can continue, however, the hitchhik...",1,Creepshow 2,The Human Tornado,0.000000,True
12887,Roosevelt reads a letter he received from Rais...,1,The Wind and the Lion,Madonna: Truth or Dare,0.010213,True
32119,"A rival arms dealer known as Falcon, who has b...",1,Lone Wolf McQuade,Armed Response,0.013306,True
159656,The soldier knocks her down and attempts to ra...,0,Primeval,Risen,0.013955,True
30698,The scientists in charge of the experiment soo...,1,Timerider: The Adventure of Lyle Swann,Out on a Limb,0.014621,True


In [39]:
### Failure Analysis contd.. ###

#Count how many times each true Title was mispredicted
wrong_counts = failures["Title"].value_counts().reset_index()
wrong_counts.columns = ["Title", "Wrong_Prediction_Count"]
wrong_counts.head()

,Title,Wrong_Prediction_Count
0,Stone,27
1,Movie 43,20
2,Fresh,15
3,Trick 'r Treat,13
4,Music Box,12


In [40]:
### Failure Analysis contd.. ###
## Systematic Error ## 

# Movie Stone has 20 wrong predictions. Looking into all 'Stone' movie predictions
test_df5[test_df5.Title=="Stone"].groupby("Predicted_Title").size()


Predicted_Title
 The Painted Veil    26
Snowden               1
Stone                56
dtype: int64

In [49]:
# Avg score when prediction is ' The Painted Veil' for 'Stone'
avg_score1 = np.mean(failures[(failures.Title=="Stone") & (failures.Predicted_Title==' The Painted Veil')]['Confidence_Score'])

# Avg score when prediction is 'Stone' for 'Stone'

avg_score2 = np.mean(test_df5_sorted[(test_df5_sorted.Title=="Stone") & (test_df5_sorted.Predicted_Title=='Stone')]['Confidence_Score'])

# Avg score when prediction is 'Snowden' for 'Stone'

avg_score3 = np.mean(test_df5_sorted[(test_df5_sorted.Title=="Stone") & (test_df5_sorted.Predicted_Title=='Snowden')]['Confidence_Score'])

avg_score1, avg_score2, avg_score3

(np.float64(0.4568727873287423),
 np.float64(0.4576750589944029),
 np.float64(0.21539655060386761))

In [42]:
## Top words/features that push the model to misclassify “Stone” as “The Painted Veil”

#Filter misclassified “Stone” examples
failures_stone = test_df5[
    (test_df5['Title'] == 'Stone') & (test_df5['Prediction_Wrong'])
].copy()

#Extract the feature vectors
preprocessor = pipeline.named_steps['preprocess']
X_failures_vec = preprocessor.transform(failures_stone[["combined_text", "released_before_2000"]])

#Extract the LinearSVC model and feature names
model = pipeline.named_steps['model']
feature_names = preprocessor.get_feature_names_out()  # TF-IDF + passthrough features

#Get the coefficients for the class that was wrongly predicted
# Get index of the wrongly predicted class
wrong_class = ' The Painted Veil'
wrong_class_int = le_wrapper.transform([wrong_class])[0]
wrong_class_idx = list(model.classes_).index(wrong_class_int)

# Coefficients for that class (LinearSVC is one-vs-rest)
coefs = model.coef_[wrong_class_idx]

#Compute contributions for each feature in the misclassified rows
# If X_failures_vec is sparse

contributions = X_failures_vec.toarray() @ coefs  # shape = [n_samples]

# Average contribution per feature across all failures
avg_contrib_per_feature = np.mean(X_failures_vec.toarray() * coefs, axis=0)

#Create a ranked table
feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Avg_Contribution': avg_contrib_per_feature
})

# Sort by absolute contribution to see top words pushing toward wrong class
feature_importance['Abs_Contribution'] = feature_importance['Avg_Contribution'].abs()
feature_importance_sorted = feature_importance.sort_values(by='Abs_Contribution', ascending=False)

# Display top 20 features
print(feature_importance_sorted.head(20)[['Feature', 'Avg_Contribution']])


              Feature  Avg_Contribution
648     tfidf__curran          0.936060
1984    tfidf__norton          0.346670
851     tfidf__edward          0.212520
1443      tfidf__john          0.157252
2719  tfidf__thriller         -0.027138
802      tfidf__drama          0.019715
1928   tfidf__mystery         -0.013458
1972      tfidf__niro         -0.009899
2317    tfidf__robert         -0.005395
1831      tfidf__meet          0.003619
264     tfidf__better          0.003017
1391      tfidf__jack         -0.002544
2696     tfidf__tells         -0.002409
2961      tfidf__work          0.001528
1697      tfidf__love          0.001313
748        tfidf__did          0.001224
1646      tfidf__life          0.001088
2626     tfidf__stone         -0.001067
149       tfidf__asks          0.000991
2836     tfidf__visit         -0.000678


In [43]:
### Failure Analysis contd.. ###
## Random Error ###

#Movie 43
# Movie 'Movie 43' has 20 wrong predictions. Looking into all 'Movie 43' predictions
test_df5[test_df5.Title=="Movie 43"].groupby("Predicted_Title").size()

Predicted_Title
 Jay and Silent Bob Strike Back    1
 What a Girl Wants                 2
 Yours, Mine and Ours              3
Darling Companion                  1
Dolphin Tale 2                     1
Going the Distance                 1
Kate & Leopold                     1
Men Don't Leave                    1
Movie 43                           1
Neighbors                          1
Our Son, the Matchmaker            1
Stand Up Guys                      1
Terminator Salvation               1
The Final Sanction                 1
The Mother of Tears                2
The Reluctant Fundamentalist       1
When Marnie Was There              1
dtype: int64

In [39]:
### Failure Analysis contd.. ###
## Random Error ###

# Avg confidence score for Movie 43 predictions
np.mean(test_df5_sorted[test_df5_sorted.Title=='Movie 43']['Confidence_Score'])

np.float64(0.14481859899360802)

In [45]:
### Failure Analysis contd.. ###
## Random Error ###

# avg confidence score for each Movie 43 prediction
avg_conf_by_pair = (
    test_df5_sorted
    .groupby(['Title', 'Predicted_Title'])['Confidence_Score']
    .mean()
    .reset_index()
    .sort_values('Confidence_Score', ascending=False)
)

avg_conf_by_pair[avg_conf_by_pair.Title== 'Movie 43']

,Title,Predicted_Title,Confidence_Score
6877,Movie 43,"Our Son, the Matchmaker",0.234967
6882,Movie 43,The Reluctant Fundamentalist,0.224738
6868,Movie 43,What a Girl Wants,0.181965
6879,Movie 43,Terminator Salvation,0.176822
6872,Movie 43,Going the Distance,0.171024
6873,Movie 43,Kate & Leopold,0.159612
6867,Movie 43,Jay and Silent Bob Strike Back,0.159362
6875,Movie 43,Movie 43,0.151159
6869,Movie 43,"Yours, Mine and Ours",0.127225
6876,Movie 43,Neighbors,0.126411


In [46]:
### Failure Analysis contd... ###
## Edge Error ##

#Checking Titles that appears only once in the Test_df5 dataset
titles_once_df = test_df5_sorted.groupby('Title').filter(lambda x: len(x) == 1)
titles_once_df

,combined_text,released_before_2000,Title,Predicted_Title,Confidence_Score,Prediction_Wrong
105026,But after she puts her studies on hold to find...,1,Living Out Loud,Birth of The Beatles,0.017447,True
70971,"Harry is a natural-born liar who, because of h...",1,Man Trouble,Man Trouble,0.035258,False
29294,"San Francisco-based Dashiell Hammett, trying t...",1,Hammett,A Time of Destiny,0.037129,True
47269,"Teen sisters Ruth and Lucille, raised by a gra...",1,Housekeeping,"Elvira, Mistress of the Dark",0.041512,True
53725,"Wanting to get a leg up on each other, they al...",1,Spellcaster,The Thirteenth Year,0.047900,True
...,...,...,...,...,...,...
153866,"When it premiered at Sundance, the film's rati...",0,This Film Is Not Yet Rated,This Film Is Not Yet Rated,0.518897,False
154616,10 MPH follows the progress of Caldwell as he ...,0,10 MPH,10 MPH,0.525270,False
159248,"Months earlier she met a young, charismatic, t...",0,Only for You,Only for You,0.543061,False
141945,[1] comedy Johnny Knoxville Katrina Holden Bro...,0,Daltry Calhoun,Daltry Calhoun,0.551957,False


In [47]:
### Failure Analysis contd... ###
## Edge Error ##

#Checking Training examples of 'Living Out Loud'
train_df[train_df.Title=='Living Out Loud'] # Only 5! Edge case

,combined_text,released_before_2000,Title
105030,"Her friend Liz Bailey, who sings at a nightclu...",1,Living Out Loud
105029,Yet what he wishes to pursue as a romantic rel...,1,Living Out Loud
105027,"Depressed, she holes up in her apartment, wher...",1,Living Out Loud
105028,"He is as lonely as she is, beset with gambling...",1,Living Out Loud
105025,Judith Moore had what she thought was a perfec...,1,Living Out Loud
